# slv_clientes
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · entidad Clientes |
| Propósito | bronze.customers_raw → silver.clientes (SCD1) y silver.clientes_historial (SCD2 sobre segmento), con emails y ciudades normalizados. Por qué SCD2 : cada venta se atribuye al segmento que el cliente tenía al comprar. Solo `segmento` abre versión: corregir un teléfono no es un cambio de negocio. |
| Reglas | reglas_calidad.REGLAS["clientes"] |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.2 · 2026-10-21 · el descarte pasa a la vista validado_clientes: limpio_ ya no descarta, así la cuarentena recibe las filas inválidas |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun import (TECNICAS, ciudad_canonica, con_secuencia, email_canonico, leer_bronze,
                   normalizar_email, parametros_cdc)
from reglas_calidad import REGEX_EMAIL, reglas

# AUTO CDC: nombre actual de la API; apply_changes es el nombre anterior.
aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Vista de limpieza `limpio_clientes`
Lee los cambios de bronze **como stream** (solo lo nuevo en cada corrida) y:
1. agrega el **orden de los cambios** (`_secuencia`: fecha efectiva, `_lsn`, `_seqval`);
2. **renombra** a snake_case en español y **normaliza** valores;
3. aplica las reglas `fallar` (detiene el pipeline) y `advertir` (entra y queda la métrica).

Aquí **no se descarta nada**: esta vista la leen dos rutas, la validación (sección 3) y la cuarentena (`slv_cuarentena`). Un `expect_all_or_drop` en una vista se aplica a todo el que la lee, así que si descartara aquí, la cuarentena nunca vería las filas inválidas.

In [ ]:
@dlt.view(name="limpio_clientes", comment="Clientes de bronze renombrados y normalizados (sin descarte).")
@dlt.expect_all_or_fail(reglas("clientes", "fallar"))      # rompe el modelo → detiene el pipeline
@dlt.expect_all(reglas("clientes", "advertir"))            # mejorable → entra y queda la métrica
def limpio_clientes():
    df = con_secuencia(leer_bronze("customers_raw"))
    email_norm = normalizar_email("Email")
    return df.select(
        F.col("CustomerID").alias("id_cliente"),
        F.trim("FirstName").alias("nombre"),
        F.trim("LastName").alias("apellido"),
        F.col("Email").alias("email_original"),
        email_norm.alias("email_normalizado"),
        email_canonico(email_norm).alias("email_canonico"),     # sin puntos en la parte local
        F.col("Phone").alias("telefono"),
        F.col("City").alias("ciudad_original"),
        ciudad_canonica("City").alias("ciudad"),
        F.col("CountryCode").alias("codigo_pais"),
        F.lower("Segment").alias("segmento"),
        F.col("SignupDate").alias("fecha_alta"),
        F.col("IsActive").alias("es_activo"),
        F.col("CreatedAt").alias("creado_en"),
        F.col("UpdatedAt").alias("actualizado_en"),
        F.coalesce(email_norm.rlike(REGEX_EMAIL), F.lit(False)).alias("es_email_valido"),  # sin email → False
        *TECNICAS)

## 3. Vista de validación `validado_clientes`
La puerta de entrada a silver: aplica las reglas `descartar` y solo deja pasar las filas que las cumplen. Lo que descarta lo recoge `slv_cuarentena` desde `limpio_clientes`, con las **mismas reglas**: cada fila termina en silver o en la cuarentena, nunca en las dos ni en ninguna.

In [ ]:
@dlt.view(name="validado_clientes", comment="Clientes que cumplen las reglas de descarte: lo que entra a silver.")
@dlt.expect_all_or_drop(reglas("clientes", "descartar"))   # inválido → no entra (lo recoge la cuarentena)
def validado_clientes():
    return dlt.read_stream("limpio_clientes")

## 4. Tabla(s) silver con AUTO CDC: `clientes`, `clientes_historial`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD1** → estado actual (una fila por clave).
- **SCD2** → una versión por cada cambio de las columnas indicadas, con su vigencia en `__START_AT` / `__END_AT`.

In [ ]:
dlt.create_streaming_table(name="clientes", comment="Clientes: estado actual (SCD1) desde CDC de Azure SQL.")
aplicar_cdc(target="clientes", source="validado_clientes", **parametros_cdc("id_cliente"))
dlt.create_streaming_table(name="clientes_historial",
                           comment="Clientes: historial SCD2 del segmento. Vigencia en __START_AT / __END_AT.")
aplicar_cdc(target="clientes_historial", source="validado_clientes", **parametros_cdc("id_cliente", historial_de=["segmento"]))
# Lo que no cumple una regla de descarte lo recoge slv_cuarentena.py.